In [2]:
!pip install mne

In [3]:
import numpy as np
import mne

from mne.decoding import CSP

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report

CHB-MIT SCALP EEG DATASET

In [4]:
# raw03 = mne.io.read_raw_edf("chb01_03.edf", preload=True)
# raw03.plot()

In [5]:
# raw03.info

In [6]:
# print(open("chb01-summary.txt").read())

In [7]:
# raw04 = mne.io.read_raw_edf("chb01_04.edf", preload=True)
# raw04.plot()

In [8]:
# raw04.info

In [9]:
# epochs = mne.make_fixed_length_epochs(raw03, duration=2)

In [10]:
# y = []
# for event in epochs.events:
#     sample = event[0]

#     epoch_start = sample / raw03.info['sfreq']
#     epoch_end = epoch_start + 2


#     seizure_start = 2996
#     seizure_end = 3036

#     if(epoch_start < seizure_end) and (epoch_end > seizure_start):
#        y.append(1)
#     else:
#        y.append(0)

# y = np.array(y)

In [11]:
# print(np.unique(y, return_counts=True))

In [12]:
# y = epochs.events[:, 2]
# print(y)

In [13]:
# from mne.decoding import CSP

# csp = CSP(n_components=4)
# X = epochs.get_data()
# X_csp = csp.fit_transform(X, y)

In [14]:
# def create_sequences(X, y, seq_len=10):
#     X_seq = []
#     y_seq = []

#     for i in range(len(X) - seq_len):
#         seq = X[i:i+seq_len]
#         label_window = y[i:i+seq_len]

#         if np.sum(label_window) >= 3:
#             label = 1
#         else:
#             label = 0

#         X_seq.append(seq)
#         y_seq.append(label)

#     X_seq = np.array(X_seq)
#     y_seq = np.array(y_seq)
#     return X_seq, y_seq

In [15]:
# X_seq, y_seq = create_sequences(X_csp, y, seq_len=10)
# n_samples, seq_len, n_features = X_seq.shape
# X_seq_flat = X_seq.reshape(n_samples, seq_len * n_features)

# X_train, X_test, y_train, y_test = train_test_split(
#     X_seq_flat, y_seq, test_size=0.2, random_state=42
# )

# scaler = StandardScaler()
# X_train = scaler.fit_transform(X_train)
# X_test = scaler.transform(X_test)

# model = LogisticRegression(class_weight='balanced')
# model.fit(X_train, y_train)
# preds = model.predict(X_test)
# cm = confusion_matrix(y_test, preds)
# print(cm)
# print(classification_report(y_test, preds))

In [16]:
# print(X_seq.shape)
# print(y_seq.shape)

In [17]:
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import confusion_matrix, classification_report

# X_train, X_test, y_train, y_test = train_test_split(
#     X_csp, y, test_size=0.2, random_state=42
# )

# model = LogisticRegression(class_weight='balanced')
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
# model.fit(X_train_scaled, y_train)
# preds = model.predict(X_test_scaled)
# cm = confusion_matrix(y_test, preds)
# print(cm)
# print(classification_report(y_test, preds))



In [18]:
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import StandardScaler
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import confusion_matrix, classification_report

# X_train, X_test, y_train, y_test = train_test_split(
#     X_csp, y, test_size=0.2, random_state=42
# )

# model = LogisticRegression(class_weight='balanced')
# model.fit(X_seq, y_seq)
# preds = model.predict(X_seq)
# cm = confusion_matrix(y_seq, preds)
# print(cm)
# print(classification_report(y_seq, preds))

In [19]:
def get_seizure_times(summary_text, target_file):
    lines = summary_text.splitlines()
    seizure_times = []
    in_target = False

    for i, line in enumerate(lines):
        line = line.strip()

        if line.startswith("File Name:"):
            current_file = line.split(":")[1].strip()
            in_target = (current_file == target_file)

        if in_target:
            if "Number of Seizures in File: 0" in line:
                return []
            if "Seizure Start Time:" in line:
                start = int(line.split(":")[1].split()[0])
                end = int(lines[i+1].split(":")[1].split()[0])
                seizure_times.append((start, end))

    return seizure_times

In [20]:
def process_file(file_path, seizure_times):
    raw = mne.io.read_raw_edf(file_path, preload=True)

    epochs = mne.make_fixed_length_epochs(raw, duration=2)
    data = epochs.get_data()

    labels = []

    for t in epochs.events[:, 0] / raw.info['sfreq']:
        label = 0
        for start, end in seizure_times:
            epoch_start = t
            epoch_end = t + 2

            if (epoch_start <= end) and (epoch_end >= start):
                label = 1
        labels.append(label)

    return data, np.array(labels)

In [21]:
# Load summary file
with open("chb01-summary.txt", "r") as f:
    summary_text = f.read()

# # Files
# file_03 = "chb01_03.edf"
# file_04 = "chb01_04.edf"

# # Get seizure times
# times_03 = get_seizure_times(summary_text, file_03)
# times_04 = get_seizure_times(summary_text, file_04)
# print(times_03)
# print(times_04)

# # Process files
# X_train_raw, y_train = process_file(file_03, times_03)
# # print(X_train_raw, y_train)
# X_test_raw, y_test = process_file(file_04, times_04)
# # print(X_test_raw, y_test)

In [22]:
X_all = []
y_all = []

files = ["chb01_03.edf", "chb01_04.edf"]

for file in files:
    seizure_times = get_seizure_times(summary_text, file)
    X, y = process_file(file, seizure_times)

    X_all.append(X)
    y_all.append(y)

X_all = np.concatenate(X_all)
y_all = np.concatenate(y_all)

print("Labels:", np.unique(y_all, return_counts=True))

Extracting EDF parameters from chb01_03.edf...
Setting channel info structure...
Creating raw.info structure...


/tmp/ipykernel_12321/3886054971.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Reading 0 ... 921599  =      0.000 ...  3599.996 secs...
Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 512 original time points ...
0 bad epochs dropped
Extracting EDF parameters from chb01_04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_12321/3886054971.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 512 original time points ...
0 bad epochs dropped
Labels: (array([0, 1]), array([3563,   37]))


In [23]:
# print(np.unique(y_train, return_counts=True))

In [24]:
# print("Train labels:", np.unique(y_train, return_counts=True))
# print("Test labels:", np.unique(y_test, return_counts=True))

In [25]:
# if len(np.unique(y_train)) < 2:
#     raise ValueError("❌ Only one class present. Cannot run CSP.")

In [26]:
seizure_times_03 = get_seizure_times(summary_text, "chb01_03.edf")
seizure_times_04 = get_seizure_times(summary_text, "chb01_04.edf")

In [27]:
X_train_raw, y_train = process_file("chb01_03.edf", seizure_times_03)
X_test_raw, y_test = process_file("chb01_04.edf", seizure_times_04)

Extracting EDF parameters from chb01_03.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_12321/3886054971.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 512 original time points ...
0 bad epochs dropped
Extracting EDF parameters from chb01_04.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 921599  =      0.000 ...  3599.996 secs...


/tmp/ipykernel_12321/3886054971.py:2: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(file_path, preload=True)


Not setting metadata
1800 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 1800 events and 512 original time points ...
0 bad epochs dropped


In [28]:
from mne.decoding import CSP

csp = CSP(n_components=4, reg='ledoit_wolf')

X_train_csp = csp.fit_transform(X_train_raw, y_train)
X_test_csp = csp.transform(X_test_raw)

Computing rank from data with rank=None
    Using tolerance 0.00055 (2.2e-16 eps * 23 dim * 1.1e+11  max singular value)
    Estimated rank (data): 23
    data: rank 23 computed from 23 data channels with 0 projectors
Reducing data rank from 23 -> 23
Estimating class=0 covariance using LEDOIT_WOLF
Done.
Estimating class=1 covariance using LEDOIT_WOLF
Done.


In [29]:
def create_sequences(X, y, seq_len=10):
    X_seq = []
    y_seq = []

    for i in range(len(X) - seq_len):
        seq = X[i:i+seq_len]
        label_window = y[i:i+seq_len]

        # Threshold (IMPORTANT — you can tweak this)
        if np.sum(label_window) >= 2:
            label = 1
        else:
            label = 0

        X_seq.append(seq)
        y_seq.append(label)

    return np.array(X_seq), np.array(y_seq)

In [30]:
X_train_seq, y_train_seq = create_sequences(X_train_csp, y_train)
X_test_seq, y_test_seq = create_sequences(X_test_csp, y_test)

In [31]:
n_samples, seq_len, n_features = X_train_seq.shape

X_train_flat = X_train_seq.reshape(n_samples, seq_len * n_features)

X_test_flat = X_test_seq.reshape(
    X_test_seq.shape[0],
    seq_len * n_features
)

In [32]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

In [33]:
model = LogisticRegression(class_weight='balanced')
model.fit(X_train_scaled, y_train_seq)

LogisticRegression(class_weight='balanced')

In [34]:
probs = model.predict_proba(X_test_scaled)[:, 1]

preds = (probs > 0.8).astype(int)

cm = confusion_matrix(y_test_seq, preds)
print(cm)

print(classification_report(y_test_seq, preds))

[[1635  133]
 [   7   15]]
              precision    recall  f1-score   support

           0       1.00      0.92      0.96      1768
           1       0.10      0.68      0.18        22

    accuracy                           0.92      1790
   macro avg       0.55      0.80      0.57      1790
weighted avg       0.98      0.92      0.95      1790



In [35]:
def smooth_predictions(preds, window=10):
    smoothed = []
    for i in range(len(preds)):
        start = max(0, i - window)
        end = min(len(preds), i + window)

        if np.mean(preds[start:end]) > 0.6:
            smoothed.append(1)
        else:
            smoothed.append(0)

    return np.array(smoothed)

In [36]:
preds_smoothed = smooth_predictions(preds)
cm = confusion_matrix(y_test_seq, preds_smoothed)
print(cm)

print(classification_report(y_test_seq, preds_smoothed))

[[1698   70]
 [   9   13]]
              precision    recall  f1-score   support

           0       0.99      0.96      0.98      1768
           1       0.16      0.59      0.25        22

    accuracy                           0.96      1790
   macro avg       0.58      0.78      0.61      1790
weighted avg       0.98      0.96      0.97      1790

